In [1]:
import numpy as np
import pandas as pd

In [2]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "IMDB Dataset.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

df.head()

C:\Users\USER\AppData\Local\Temp\ipykernel_16404\3431149111.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.shape

(50000, 2)

In [4]:
df.drop_duplicates(inplace=True)

In [5]:
df.shape

(49582, 2)

In [6]:
import re
def remove_tags(raw_text):
    cleaned_text=re.sub(re.compile('<.*?>'),'',raw_text)
    return cleaned_text
df['review']=df['review'].apply(remove_tags)

In [7]:
df.head(2)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming tec...,positive


In [8]:
df['review']=df['review'].apply(lambda x:x.lower())

In [9]:
df.head(1)

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive


In [10]:
from nltk.corpus import stopwords

sw_list=stopwords.words('english')
df['review']=df['review'].apply(lambda x:[item for item in x.split() if item not in sw_list]).apply(lambda x:' '.join(x))

In [11]:
df.head(2)

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production. filming technique...,positive


In [12]:
import gensim

In [13]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [14]:
story=[]

for doc in df['review']:
    raw_sent=sent_tokenize(doc)

    for sent in raw_sent:
        story.append(simple_preprocess(sent))

In [15]:
story[0:100]

[['one', 'reviewers', 'mentioned', 'watching', 'oz', 'episode', 'hooked'],
 ['right',
  'exactly',
  'happened',
  'me',
  'the',
  'first',
  'thing',
  'struck',
  'oz',
  'brutality',
  'unflinching',
  'scenes',
  'violence',
  'set',
  'right',
  'word',
  'go'],
 ['trust', 'me', 'show', 'faint', 'hearted', 'timid'],
 ['show', 'pulls', 'punches', 'regards', 'drugs', 'sex', 'violence'],
 ['hardcore',
  'classic',
  'use',
  'word',
  'it',
  'called',
  'oz',
  'nickname',
  'given',
  'oswald',
  'maximum',
  'security',
  'state',
  'penitentary'],
 ['focuses',
  'mainly',
  'emerald',
  'city',
  'experimental',
  'section',
  'prison',
  'cells',
  'glass',
  'fronts',
  'face',
  'inwards',
  'privacy',
  'high',
  'agenda'],
 ['em',
  'city',
  'home',
  'many',
  'aryans',
  'muslims',
  'gangstas',
  'latinos',
  'christians',
  'italians',
  'irish',
  'more',
  'so',
  'scuffles',
  'death',
  'stares',
  'dodgy',
  'dealings',
  'shady',
  'agreements',
  'never',
  'far

In [16]:
len(story)

528620

In [17]:
model=gensim.models.Word2Vec(
    window=10,
    min_count=2
)

In [18]:
model.build_vocab(story)

In [19]:
model.train(story,total_examples=model.corpus_count,epochs=model.epochs)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


(29411961, 30767745)

In [20]:
len(model.wv.index_to_key)

61843

In [21]:
def document_vector(doc):
    # remove out-of-vocabulary words
    doc = [word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc], axis=0)


In [22]:

document_vector(df['review'].values[0])

array([ 1.77656204e-01, -1.45181298e-01,  1.42617121e-01, -1.65533110e-01,
        5.91126144e-01, -3.05935919e-01, -7.97773451e-02,  3.03686470e-01,
        4.48330432e-01, -2.44083256e-01, -4.19118315e-01, -4.48727727e-01,
       -1.02068275e-01,  3.11583400e-01,  1.37695462e-01,  1.67700067e-01,
        6.81196898e-02, -1.18397161e-01, -5.16915619e-02, -3.37974668e-01,
        1.45469993e-01,  3.62037271e-01, -3.27798396e-01,  3.25750798e-01,
       -1.85639828e-01, -6.65781572e-02, -6.53257370e-02,  9.69033018e-02,
       -4.20497715e-01, -1.91886038e-01,  2.98176974e-01, -3.35941881e-01,
        1.93688825e-01, -3.80960941e-01, -8.89170989e-02,  5.09788215e-01,
       -7.63897002e-02,  6.45966083e-02,  2.06614077e-01, -4.75447893e-01,
       -1.88514978e-01,  9.48951691e-02, -1.52328447e-01, -3.19686323e-01,
        3.34840804e-01, -2.75709599e-01,  6.67713583e-02, -1.20563041e-02,
       -2.21652344e-01,  4.55534786e-01,  4.88388121e-01,  3.34050842e-02,
       -7.93125778e-02, -

In [23]:
from tqdm import tqdm

In [24]:

X = []
for doc in tqdm(df['review'].values):
    X.append(document_vector(doc))

  0%|▎                                                                           | 208/49582 [00:43<2:52:08,  4.78it/s]


KeyboardInterrupt: 

In [ ]:
X = np.array(X)

In [ ]:

X[0]

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

y = encoder.fit_transform(df['sentiment'])

In [ ]:

y

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [ ]:

rf = RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test,y_pred)